# Exercise 9.3 — One toolbox for explanations: DALEX (with solutions)

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day9/day6-03-dalex.html>.*

*Open it in Jupyter or VS Code and run it a cell at a time. Keep it in the folder that holds your `data` folder. This version has the worked answers filled in.*

Exercises 9.1 and 9.2 used a different package for each explanation. The
**DALEX** package puts them behind one interface. You wrap the fitted model
once in an **explainer**, then ask the explainer each question. The package
exists in R (with **DALEXtra** for tidymodels) and in Python (`dalex`), with
the same function names.

> **The plots are for looking at**
>
> The code for every plot is included, so that you can run it and see the figure. You are not expected to write it yourself.

### Packages and data

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn "dalex>=1.8"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import dalex as dx             # explanations for any model

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

The data are the 750 people in
[`data/diabetes_clean.csv`](data/diabetes_clean.csv), as in Exercises 9.1 and
9.2. The outcome is `diabetes`, coded 1 for `pos` and 0 for `neg`; the
predictors are pregnant, glucose, mass, pedigree and age, in their own units.
As always, 80% of the people are used for training and 20% are held back as
the test set.

In [ ]:
diabetes = pd.read_csv("data/diabetes_clean.csv")
diabetes = diabetes[["pregnant", "glucose", "mass", "pedigree", "age", "diabetes"]]
diabetes["diabetes"] = (diabetes["diabetes"] == "pos").astype(int)       # 1 = pos, 0 = neg

predictors = ["pregnant", "glucose", "mass", "pedigree", "age"]

train, test = train_test_split(diabetes, test_size=0.2, random_state=42)

------------------------------------------------------------------------

### Part 1 — The model and the explainer

The black box is the random forest of Exercises 9.1 and 9.2. Fit it on the
training rows:

In [ ]:
rf = RandomForestClassifier(n_estimators=500, random_state=1)
rf = rf.fit(train[predictors], train["diabetes"])

Evaluate it once on the test set:

In [ ]:
test_class = rf.predict(test[predictors])                # predicted class, threshold 0.5
test_prob  = rf.predict_proba(test[predictors])[:, 1]    # probability of diabetes

print("accuracy:", round(accuracy_score(test["diabetes"], test_class), 3))
print("AUC:     ", round(roc_auc_score(test["diabetes"], test_prob), 3))

Accuracy 0.713 and AUC 0.794 on the 150 test patients.

Now wrap the model in an explainer. `dx.Explainer()` needs the model, the
predictors of the people to explain it on (the test patients), and the true
outcome `y` **as numbers, 1 for diabetes and 0 for none**.

> **Stuck?**
>
> `data` is the table of predictors only, without the outcome column: the same
> columns the forest was fitted on.

> **Worked answer**

In [ ]:
explainer = dx.Explainer(
    rf,
    data=test[predictors],
    y=test["diabetes"],                           # already 1 = diabetes, 0 = none
    label="Random forest",
    verbose=False,                                # no progress report
)

A quick check that the explainer is set up right: its AUC should be the one
you just computed.

In [ ]:
explainer.model_performance().result.round(3)

It is, 0.794.

------------------------------------------------------------------------

### Part 2 — Which variables matter?

`model_parts()` is the **permutation importance** of Exercise 9.1: shuffle one
variable in the test set, see how much worse the model gets, repeat 10 times
and average. DALEX measures "worse" as $1 - \text{AUC}$, so a larger number is
a worse model. `_full_model_` is the value with nothing shuffled.

In [ ]:
importance_dalex = explainer.model_parts(random_state=1)

importance_dalex.result[["variable", "dropout_loss"]].round(3)

With nothing shuffled the loss is 0.206. Shuffling glucose raises it to
0.352; shuffling any other variable raises it by 0.036 at most (age), and
shuffling pedigree changes it by less than 0.001. Glucose is by far the most
important variable, as in Exercise 9.1, and here it took one line of code.

------------------------------------------------------------------------

### Part 3 — How does the prediction change with glucose?

An importance says *how much* a variable matters, not *how*. For one patient,
set her glucose to 73, then a little higher, and so on up to 196, keep everything else about
her as it is, and predict each time. The curve you get is her **ICE profile**
(individual conditional expectation). The average of the 150 ICE profiles is
the **partial dependence profile**: what the model predicts, on average, at
each glucose level. `model_profile()` computes both.

> **Stuck?**
>
> Give the name of the variable as a string, in quotes, inside the list.

> **Worked answer**

In [ ]:
pdp_glucose = explainer.model_profile(variables=["glucose"], type="partial",
                                      verbose=False)

In [ ]:
ice = pdp_glucose.raw_profiles.result

fig, ax = plt.subplots(figsize=(6, 3.5))
for patient_id in ice["_ids_"].unique():
    one = ice[ice["_ids_"] == patient_id]
    ax.plot(one["glucose"], one["_yhat_"], color="grey", alpha=0.3, linewidth=0.7)
ax.plot(pdp_glucose.result["_x_"], pdp_glucose.result["_yhat_"], color="steelblue", linewidth=2.5)
ax.set_xlabel("Glucose (mg/dL)")
ax.set_ylabel("Predicted probability of diabetes")
ax.set_title("ICE profiles (grey) and partial dependence (blue)")
fig.tight_layout()
plt.show()

**Question!** Describe the blue curve in words. Do all the grey curves have
the same shape?

Below a glucose of about 100 the average prediction stays low, between 0.15
and 0.20. It climbs gradually to about 0.37 at 140, rises steeply between 145
and 157 to about 0.66, and stays between 0.63 and 0.67 above that. The steps
come from the forest's splits. Every grey curve ends higher than it
starts, but they start at different heights (from 0.04 to 0.64) and rise by
very different amounts (from 0.08 to 0.85). How much glucose matters depends on a patient's other values;
the average curve hides that.

------------------------------------------------------------------------

### Part 4 — One patient

`predict_parts()` explains one prediction. Take the first test patient:

In [ ]:
new_observation = test[predictors].iloc[[0]]

print(new_observation)
print(explainer.predict(new_observation))     # her predicted probability

Ask for her **SHAP values**, as in Exercise 9.2. DALEX estimates them from 25
random orders of the variables, with the test patients as the background.

> **Stuck?**
>
> The type is a string, the name of the method in lower case. See
> `help(explainer.predict_parts)`.

> **Worked answer**

In [ ]:
shap_patient = explainer.predict_parts(new_observation, type="shap", B=25,
                                       random_state=1)

shap_average = shap_patient.result[shap_patient.result["B"] == 0]   # B = 0 holds the average of the 25 orders
shap_average = shap_average[["variable", "contribution"]]
shap_average.sort_values("contribution", key=abs, ascending=False).round(3)

This split puts a different person first than the R split does. She is 35,
with a glucose of 115, and the forest gives her 0.276. The average prediction
over the test patients is 0.356. Her glucose pulls the prediction down by
0.144; each of the other four variables moves it by 0.036 or less. As in
Exercise 9.2, the contributions add up from the average to her prediction.
With `type="break_down"` DALEX adds the variables in one fixed order instead
of averaging over many.

> **Key points**
>
> - Wrap the fitted model once in an explainer, with the test patients and `y` coded 1 for diabetes and 0 for none. Every explanation is then one function call.
> - `model_parts()` is the permutation importance of Exercise 9.1: glucose matters most.
> - `model_profile()` gives the partial dependence profile (the average) and the ICE profiles (one per patient) for one variable. Read both: the average can hide patients who differ.
> - `predict_parts()` explains one patient, with SHAP values as in Exercise 9.2.